# Artisanal Mining Site Suitability Scanner
## Ashanti Region, Ghana

This notebook builds an automated geospatial pipeline that scores land 
suitability for artisanal mining across the Ashanti Region using satellite 
data and terrain analysis.

**Inputs:**
- Terrain slope derived from SRTM Digital Elevation Model (30m resolution)
- Vegetation cover (NDVI) from Landsat 9 satellite imagery (2023)
- Proximity to known mining footprints (Global Mining Polygons dataset)

**Output:** A 0–3 suitability score map where higher scores indicate 
flatter terrain, lower vegetation, and proximity to existing mining activity.

**Tools:** Python · Google Earth Engine · geemap · GeoPandas · Landsat 9

## 1. Setup & Authentication

In [1]:
import ee
import geemap
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project='gis-learning-project-487822')
m = geemap.Map()

## 2. Study Area — Ashanti Region Boundary

Load the Ghana administrative boundaries (GADM Level 1) and filter 
to the Ashanti Region. Convert to an Earth Engine object for spatial 
operations and clipping.

In [2]:
# Boundary - Ashanti Region
ghana = gpd.read_file("data/gadm41_GHA_shp/gadm41_GHA_1.shp")
ashanti_gdf = ghana[ghana['NAME_1'] == 'Ashanti']
ashanti_ee = geemap.gdf_to_ee(ashanti_gdf)
m.addLayer(ashanti_ee, {}, 'Ashanti Boundary')

## 3. Terrain Analysis — Elevation & Slope

Load the USGS SRTM Digital Elevation Model (30m resolution) and derive 
slope in degrees. Slope is a key suitability factor — flatter terrain 
(< 8°) is more accessible for artisanal mining operations.

In [3]:
# Elevation & Slope
dem = ee.Image("USGS/SRTMGL1_003").clip(ashanti_ee)
slope = ee.Terrain.slope(dem) # Calculate slope in degrees
m.addLayer(dem, {'min': 0, 'max': 1000}, 'Elevation (DEM)')
m.addLayer(slope, {'min': 0, 'max': 30, 'palette': ['white', 'black']}, 'Slope')

## 4. Vegetation Cover — Landsat 9 NDVI

Filter Landsat 9 Collection 2 imagery for 2023 with less than 20% cloud 
cover. Compute a median composite to reduce cloud noise, then calculate 
NDVI (Normalised Difference Vegetation Index).

Lower NDVI (< 0.4) indicates sparse vegetation or bare ground — more 
accessible for surface mining with less clearing required.

In [4]:
# Landsat 9 / NDVI
collection = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
              .filterBounds(ashanti_ee)
              .filterDate('2023-01-01', '2023-12-31')
              .filter(ee.Filter.lessThan('CLOUD_COVER', 20)))

image = collection.median().clip(ashanti_ee)
ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
ndvi_vis = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}
m.addLayer(ndvi, ndvi_vis, 'Vegetation (NDVI)')

## 5. Existing Mining Footprints

Load the Global Mining Polygons dataset (sat-io) from Earth Engine 
and filter to the Ashanti Region. These polygons represent known active 
and historical mining sites — used to define proximity zones.

In [5]:
# Mining Footprints - Load global mining polygons dataset from Earth Engine community catalog
mining_polygons = ee.FeatureCollection("projects/sat-io/open-datasets/global-mining/global_mining_polygons")
ashanti_mines = mining_polygons.filterBounds(ashanti_ee)
m.addLayer(ashanti_mines, {'color': 'red'}, 'Actual Mining Footprints')

## 6. Suitability Scoring

Each pixel in the Ashanti Region is scored across three criteria:

| Criterion | Condition | Score |
|---|---|---|
| Slope | < 8° (flat/accessible) | 1 |
| NDVI | < 0.4 (low vegetation) | 1 |
| Proximity | Within 5km of known mine | 1 |

**Total score range: 0 (low) to 3 (high suitability)**

Pixels with no data are unmasked to 0 (conservative assumption).

In [6]:
# Step 1 — Slope score: 1 if slope < 8 degrees, else 0
slope_score = slope.unmask(0).lt(8).rename('slope_score')

# Step 2 — NDVI score: 1 if NDVI < 0.4 (sparse vegetation), else 0
ndvi_score = ndvi.unmask(0).lt(0.4).rename('ndvi_score')

# Step 3 — Proximity score: 1 if within 5km of a known mining footprint
# Buffer the mining geometries by 5000m, paint value 1 inside, 0 outside
mines_proximity = ashanti_mines.geometry().buffer(5000)
proximity_score = (ee.Image.constant(0)
                  .paint(mines_proximity, 1)
                  .clip(ashanti_ee)
                  .unmask(0)
                  .rename('proximity_score'))

# Step 4 — Combine all scores
suitability = (slope_score
               .add(ndvi_score)
               .add(proximity_score)
               .clip(ashanti_ee)
               .rename('suitability'))

## 7. Visualisation & Output

Display the suitability map with a red–yellow–green colour ramp 
at 70% opacity so the mining footprints remain visible underneath. 
Add a legend and title for a clean, presentation-ready output.

In [7]:
# Colour ramp: red = low suitability, green = high suitability
suitability_vis = {
    'min': 0, 'max': 3,
    'palette': ['#d7191c', '#fdae61', '#ffffbf', '#1a9641']
}
m.addLayer(suitability, suitability_vis, 'Suitability Score', True, 0.7)
m.centerObject(ashanti_ee, 8)

# Add legend
legend_dict = {
    'High suitability (score 3)':    '#1a9641',
    'Moderate-high (score 2)':       '#ffffbf',
    'Moderate-low (score 1)':        '#fdae61',
    'Low suitability (score 0)':     '#d7191c',
    'Actual mining footprints':      'ff0000'
}

m.add_legend(
    title='Mining Site Suitability',
    legend_dict=legend_dict,
    position='bottomleft'
)

# Add map title
m.add_text(
    text='Artisanal Mining Site Suitability Scanner — Ashanti Region, Ghana',
    position='bottomright',
    font_size=14
)

m

Map(center=[6.799268449954375, -1.4572414329374803], controls=(WidgetControl(options=['position', 'transparent…

## 8. Export Interactive Map

In [8]:
# Export as a standalone interactive HTML file for sharing
m.to_html('ashanti_mining_suitability.html')
print("Map exported to ashanti_mining_suitability.html")

Map exported to ashanti_mining_suitability.html
